In [75]:
#Import modules
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy
import os

#read data
state_filepath='/Users/melodyqian/Documents/GitHub/FindMyNuclearWaste/data/State_Level_Demographics_filtered.csv'
statedf=pd.read_csv(state_filepath)
site_filepath='/Users/melodyqian/Documents/GitHub/FindMyNuclearWaste/data/MasterDataset.csv'
sitedf=pd.read_csv(site_filepath)

#Clean
sitedf= sitedf[sitedf['county_data']!=1]
sitedf=sitedf[sitedf['Type']!= 'Government Facility/Multiple']



statecolumns=['state_abbrev','white_percent', 'black_percent', 'asian_percent', 'pacific_percent', 'native_percent', 'hispanic_percent', 'nonhispanic_percent']
sitecolumns=['state_abbrev','white_percent', 'black_percent', 'asian_percent', 'pacific_percent', 'native_percent', 'hispanic_percent', 'nonhispanic_percent']

In [15]:
statedf=statedf[statecolumns]
sitedf=sitedf[sitecolumns]

In [22]:
mergedf= sitedf.merge(statedf, on='state_abbrev',suffixes=('_site','_state'))
racetncolumns=[col for col in statedf.columns if col!='state_abbrev']
lqdf=pd.DataFrame()
for race in racetncolumns:
    lqdf[f'{race}_lq'] = mergedf[f'{race}_site'] / mergedf[f'{race}_state']


In [ ]:
meltedlq = lqdf.melt(var_name='Demographic', value_name='Value')
medianlq = meltedlq.groupby('Demographic')['Value'].median().reset_index()
meanlq = meltedlq.groupby('Demographic')['Value'].mean().reset_index()

                Demographic     Value
0          white_percent_lq  1.604538
1          white_percent_lq  1.205412
2          white_percent_lq  1.230012
3          white_percent_lq  0.632091
4          white_percent_lq  1.507293
..                      ...       ...
450  nonhispanic_percent_lq  1.021242
451  nonhispanic_percent_lq  1.042945
452  nonhispanic_percent_lq  1.165644
453  nonhispanic_percent_lq  1.155718
454  nonhispanic_percent_lq  1.661958

[455 rows x 2 columns]


In [45]:
rename_map = {
    'white_percent_lq': 'White',
    'black_percent_lq': 'Black',
    'asian_percent_lq': 'Asian',
    'pacific_percent_lq': 'Pacific Islander',
    'native_percent_lq': 'Native',
    'hispanic_percent_lq': 'Hispanic',
    'nonhispanic_percent_lq': 'Non-Hispanic'
}


In [48]:
medianlq['Demographic'] = medianlq['Demographic'].replace(rename_map)
meanlq['Demographic'] = meanlq['Demographic'].replace(rename_map)
order = ['White', 'Black', 'Asian', 'Pacific Islander', 'Native', 'Hispanic', 'Non-Hispanic']

In [49]:
# Sort
medianlq['Demographic'] = pd.Categorical(medianlq['Demographic'], categories=order, ordered=True)
medianlq = medianlq.sort_values('Demographic').reset_index(drop=True)

meanlq['Demographic'] = pd.Categorical(meanlq['Demographic'], categories=order, ordered=True)
meanlq = meanlq.sort_values('Demographic').reset_index(drop=True)

In [85]:
figmean= go.Figure()
figmean.add_trace(go.Bar(
    x=meanlq['Demographic'],
    y=meanlq['Value'],
    name='State',
    #marker_color='rgb(211, 228, 36)',
    marker=dict(
        color=meanlq['Value'],
        colorscale='YlGn'),
    #text=mstate['hover_text'],
    textposition='none'
))
figmean.add_hline(
    y=1,
    line_color='Red',
    layer='below',
    annotation_text='Equal Proportion in Site and State',
    annotation_position='top',
    annotation_font_color='Red'
)
 
figmean.update_layout(
    title_text = 'Racial and Ethnic Location Quotient',
    width=900,
    xaxis_title='<b>Race/Ethnicity<b>',
    yaxis_title='<b>Location Quotient<b>',
)
os.chdir('/Users/melodyqian/Documents/GitHub/FindMyNuclearWaste/Visuals')
figmean.show()
#figmean.write_html('RacetnLQ.html')